<div style='background:#0b0e10;color:#f4f5f6;border:1px solid #2c3136;border-radius:18px;padding:30px 24px;text-align:center'><div style='font-family:monospace;font-size:12px;letter-spacing:.22em;color:#9aa1aa'>OFFICIAL COLAB · v1.4.2</div><h1 style='font-size:38px;margin:12px 0 4px'>MINEIRO</h1><p style='font-size:18px;color:#a3a9b0;margin:0'>OSINT Intelligence Workbench</p><p style='font-size:13px;color:#7f8790;margin-top:14px'>Collection → Evidence → Correlation → Assessment</p></div>

**Uso:** Runtime → Run all. Ao final, abra o link exibido em **Mineiro UI**.

> O notebook executa a aplicação na VM temporária do Colab. Ele não é hospedagem permanente.

## 1 · Environment
Configuração canônica. Normalmente você não precisa alterar nada.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

REPO_URL = 'https://github.com/Ridd1kulusC0d3r/Mineiro-OSINT-Extractor.git'
BRANCH = 'main'
ROOT = Path('/content/Mineiro-OSINT-Extractor')
PORT = 3000
LOG_FILE = Path('/content/mineiro-server.log')
NOTEBOOK_VERSION = '1.4.2'

display(HTML(f"""
<div style='border:1px solid #ddd;border-radius:12px;padding:14px 16px'>
<b>Mineiro Official Colab</b><br>
<span style='color:#666'>branch: {BRANCH} · port: {PORT} · notebook: {NOTEBOOK_VERSION}</span>
</div>"""))

## 2 · Bootstrap
Clona ou atualiza o projeto, garante Node.js 22+, instala dependências e valida o Registry.

In [ ]:
import os, subprocess, re

def run(cmd, cwd=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if ROOT.exists():
    run(['git', 'fetch', 'origin'], cwd=ROOT)
    run(['git', 'checkout', BRANCH], cwd=ROOT)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT)
else:
    run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(ROOT)])

node_version = subprocess.check_output(['node', '--version'], text=True).strip()
major = int(re.sub(r'[^0-9].*', '', node_version.lstrip('v')))
if major < 22:
    print(f'Node {node_version} detectado. Atualizando para Node 22...')
    run(['npm', 'install', '-g', 'n'])
    run(['n', '22'])

run(['npm', 'install', '--no-audit', '--no-fund', '--prefer-offline'], cwd=ROOT)
run(['npm', 'run', 'registry:validate'], cwd=ROOT)

print('\n[OK] Ambiente preparado em', ROOT)

## 3 · Start Mineiro
Inicia o servidor, espera o health check e abre a UI pelo proxy oficial do Colab.

In [ ]:
import subprocess, time, requests
from google.colab import output
from IPython.display import display, HTML, Javascript

try:
    if 'mineiro_server' in globals() and mineiro_server.poll() is None:
        mineiro_server.terminate()
        mineiro_server.wait(timeout=10)
except Exception:
    pass

log_handle = open(LOG_FILE, 'w', buffering=1)
mineiro_server = subprocess.Popen(
    ['npm', 'run', 'dev'],
    cwd=ROOT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

health = None
for _ in range(45):
    try:
        health = requests.get(f'http://127.0.0.1:{PORT}/api/health', timeout=2)
        if health.ok:
            break
    except Exception:
        pass
    time.sleep(1)

if not health or not health.ok:
    print(LOG_FILE.read_text()[-6000:] if LOG_FILE.exists() else 'Sem log disponível.')
    raise RuntimeError('Mineiro não ficou saudável dentro do tempo esperado.')

proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
payload = health.json()
display(HTML(f"""
<div style='border:1px solid #cfd3d7;border-radius:14px;padding:18px'>
<div style='font-size:12px;color:#666;letter-spacing:.12em'>ENVIRONMENT READY</div>
<div style='font-size:20px;font-weight:700;margin:6px 0'>Mineiro UI</div>
<a href='{proxy_url}' target='_blank' style='display:inline-block;background:#111;color:#fff;padding:10px 16px;border-radius:8px;text-decoration:none'>Abrir Mineiro</a>
<div style='margin-top:10px;color:#777;font-size:12px'>health: {payload}</div>
</div>"""))
display(Javascript(f"window.open('{proxy_url}', '_blank');"))

## 4 · Validation opcional
Use para validar a demo sintética e a camada analítica sem consultar pessoas reais.

In [ ]:
run(['npm', 'run', 'demo'], cwd=ROOT)
run(['npm', 'run', 'intelligence:test'], cwd=ROOT)

## 5 · Diagnostics
Mostra as últimas linhas do servidor. Útil quando a UI não abre.

In [ ]:
if LOG_FILE.exists():
    print('\n'.join(LOG_FILE.read_text(errors='replace').splitlines()[-100:]))
else:
    print('Log ainda não existe.')

## 6 · Stop
Encerra o servidor desta sessão.

In [ ]:
if 'mineiro_server' in globals() and mineiro_server.poll() is None:
    mineiro_server.terminate()
    mineiro_server.wait(timeout=10)
    print('Mineiro encerrado.')
else:
    print('Nenhum servidor ativo nesta sessão.')